# Approach A — Notebook 2: Hyper-Fast Sharded Test Inference Worker
**Amazon ML Challenge 2026: Business Entity Resolution**

### Instructions for Parallel Execution:
- Set `SHARD_IDX` to `0`, `1`, `2`, or `3` (for 4 parallel notebook sessions).
- Or leave `RUN_ALL_SHARDS = True` to process all 4 shards sequentially in one notebook session.
- Runs in **~12–18 minutes per shard** (or under 20 minutes total if parallelized across 4 Kaggle notebooks).

**Inputs Required:**
1. Competition Test Dataset (`test_source1.tsv`)
2. Model parameters (`iso_thr.pkl` and `lgbm_fold0/1/2.txt`)
3. Pre-computed Target Index from Notebook 1 (`target_metadata.pkl` & `target_sparse_matrices.pkl`)


In [ ]:
# 0. Dependencies Setup
!pip install sparse_dot_topn rapidfuzz lightgbm pyarrow --quiet


In [ ]:
# 1. Shard & Parallel Configuration
SHARD_IDX      = 0       # Set to 0, 1, 2, or 3 for 4 parallel Kaggle sessions
NUM_SHARDS     = 4       # Total shards
RUN_ALL_SHARDS = False   # Set True if you want a single notebook to run all shards sequentially

BLOCK_TOP_K        = 50
BLOCK_MIN_SIM       = 0.12
SPARSE_CHUNK_SIZE   = 12000

import os, sys, time, glob, re, gc, pickle, unicodedata, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

N_CORES = os.cpu_count() or 8
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, str(N_CORES))
os.environ.pop("GOMP_CPU_AFFINITY", None)
os.environ.pop("KMP_AFFINITY", None)

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from sparse_dot_topn import sp_matmul_topn
import lightgbm as lgb
from rapidfuzz import fuzz, process as rf_process
from rapidfuzz.distance import Levenshtein as Lev

warnings.filterwarnings("ignore")
print(f"[Worker] Running Shard {SHARD_IDX}/{NUM_SHARDS} on {N_CORES} CPU cores")


In [ ]:
# 2. Path & Model Artifact Discovery
def find_file(filename, search_root="/kaggle/input"):
    hits = glob.glob(os.path.join(search_root, "**", filename), recursive=True)
    return hits[0] if hits else None

if os.path.isdir("/kaggle/input"):
    TEST_DIR   = os.path.dirname(find_file("test_source1.tsv") or "/kaggle/input/dataset/test/test_source1.tsv")
    MODEL_DIR  = os.path.dirname(find_file("iso_thr.pkl") or "/kaggle/input/approach_a_models/iso_thr.pkl")
    INDEX_DIR  = os.path.dirname(find_file("target_metadata.pkl") or "/kaggle/working/target_index/target_metadata.pkl")
    WORK_DIR   = "/kaggle/working"
else:
    TEST_DIR   = os.path.abspath("data/student_resource/dataset/test")
    MODEL_DIR  = os.path.abspath("models")
    INDEX_DIR  = os.path.abspath("output/approach_a/target_index")
    WORK_DIR   = os.path.abspath("output/approach_a")

OUTPUT_SHARD_DIR = os.path.join(WORK_DIR, "shard_outputs")
os.makedirs(OUTPUT_SHARD_DIR, exist_ok=True)

# Load trained models & calibration
with open(os.path.join(MODEL_DIR, "iso_thr.pkl"), "rb") as f:
    saved = pickle.load(f)
iso, t_first, t_rest = saved["iso"], saved["t_first"], saved["t_rest"]
feature_names = saved["feature_names"]

fold_files = sorted(glob.glob(os.path.join(MODEL_DIR, "lgbm_fold*.txt")))
models = [lgb.Booster(model_file=f) for f in fold_files]
print(f"Loaded {len(models)} LightGBM folds. Thresholds: t_first={t_first:.4f}, t_rest={t_rest:.4f}")

# Load target index
print(f"Loading target metadata from {INDEX_DIR}...")
with open(os.path.join(INDEX_DIR, "target_metadata.pkl"), "rb") as f:
    target_meta = pickle.load(f)

print(f"Loading pre-transformed sparse country matrices...")
with open(os.path.join(INDEX_DIR, "target_sparse_matrices.pkl"), "rb") as f:
    target_sparse_dict = pickle.load(f)

print(f"Loaded target index with {len(target_meta['entity_ids']):,} records")


In [ ]:
# 3. Load & Partition Test S1
def norm_s1_row(text):
    if not text or pd.isna(text): return "", "", ""
    t = str(text).lower().strip().replace("&", " and ")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    parts = re.split(r"dba", t)
    p = parts[0].strip() if parts else t
    trade = parts[1].strip() if len(parts) > 1 else ""
    return t, p, trade

s1_full = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="	", dtype=str).fillna("")
total_s1 = len(s1_full)
print(f"Total Test S1 entities: {total_s1:,}")


In [ ]:
# 4. Core Pipeline Functions: Blocking, Features, Inference
def run_fast_country_blocking(s1_sub, route_key, vec, top_k, min_sim):
    min_sim_strict = float(np.nextafter(np.float32(min_sim), np.float32(-np.inf)))
    text_col = "name_core" if route_key == "name" else ("addr_norm" if route_key == "addr" else "combined")
    s1_texts = s1_sub[text_col].values
    s1_countries = s1_sub["country_norm"].values

    buckets_data = target_sparse_dict[route_key]["buckets"]
    out_s1, out_s23, out_sim = [], [], []

    for c, b_info in buckets_data.items():
        if c == "__unknown__":
            s1_idx = np.where(s1_countries == "__unknown__")[0].astype(np.int32)
        else:
            s1_idx = np.where(s1_countries == c)[0].astype(np.int32)
        if len(s1_idx) == 0: continue

        s23_idx_global = b_info["idx"]
        s23_mat_T = b_info["mat_T"]

        for ci in range(0, len(s1_idx), SPARSE_CHUNK_SIZE):
            chunk_g = s1_idx[ci:ci+SPARSE_CHUNK_SIZE]
            c_mat = sk_normalize(vec.transform(s1_texts[chunk_g]))
            C = sp_matmul_topn(c_mat, s23_mat_T, top_n=top_k, threshold=min_sim_strict, sort=True, n_threads=N_CORES).tocoo()
            if len(C.row):
                out_s1.append(chunk_g[C.row.astype(np.int32)])
                out_s23.append(s23_idx_global[C.col.astype(np.int32)])
                out_sim.append(C.data.astype(np.float32))

    if out_s1:
        return np.concatenate(out_s1), np.concatenate(out_s23), np.concatenate(out_sim)
    return np.array([], dtype=np.int32), np.array([], dtype=np.int32), np.array([], dtype=np.float32)

def _binary_intersection(text1, text23, idx1, idx23, analyzer, token_pattern=None, ngram_range=(1, 1)):
    n = len(idx1)
    nonempty = pd.concat([pd.Series(text1)[pd.Series(text1) != ""], pd.Series(text23)[pd.Series(text23) != ""]], ignore_index=True)
    if nonempty.empty:
        z = np.zeros(n, dtype=np.float32)
        return z, z.copy(), z.copy()
    kwargs = dict(analyzer=analyzer, ngram_range=ngram_range, binary=True, dtype=np.float32)
    if token_pattern is not None: kwargs["token_pattern"] = token_pattern
    cv = CountVectorizer(**kwargs)
    cv.fit(nonempty)
    X1 = cv.transform(text1); X23 = cv.transform(text23)
    A, B = X1[idx1], X23[idx23]
    inter = np.asarray(A.multiply(B).sum(axis=1)).ravel().astype(np.float32)
    size1 = np.asarray(X1.sum(axis=1)).ravel()[idx1].astype(np.float32)
    size2 = np.asarray(X23.sum(axis=1)).ravel()[idx23].astype(np.float32)
    return inter, size1, size2

def _jac_ov(inter, size1, size2):
    union = size1 + size2 - inter
    jac = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
    minsz = np.minimum(size1, size2)
    ov = np.divide(inter, minsz, out=np.zeros_like(inter), where=minsz > 0)
    return jac.astype(np.float32), ov.astype(np.float32)

def _rf_worker(task):
    rows_chunk, nc1_c, nc2_c, a1_c, a2_c, bounds = task
    wr, ts, tt, pa, lv, ats, ate = [], [], [], [], [], [], []
    for start, end in bounds:
        a_name = nc1_c[start]; b_names = nc2_c[start:end].tolist()
        a_addr = a1_c[start];  b_addrs = a2_c[start:end].tolist()
        wr.append(np.array(rf_process.cdist([a_name], b_names, scorer=fuzz.WRatio, workers=1)[0], np.float32) / 100.0)
        ts.append(np.array(rf_process.cdist([a_name], b_names, scorer=fuzz.token_sort_ratio, workers=1)[0], np.float32) / 100.0)
        tt.append(np.array(rf_process.cdist([a_name], b_names, scorer=fuzz.token_set_ratio, workers=1)[0], np.float32) / 100.0)
        pa.append(np.array(rf_process.cdist([a_name], b_names, scorer=fuzz.partial_ratio, workers=1)[0], np.float32) / 100.0)
        lv.append(np.array(rf_process.cdist([a_name], b_names, scorer=Lev.normalized_similarity, workers=1)[0], np.float32))
        ats.append(np.array(rf_process.cdist([a_addr], b_addrs, scorer=fuzz.token_sort_ratio, workers=1)[0], np.float32) / 100.0)
        ate.append(np.array(rf_process.cdist([a_addr], b_addrs, scorer=fuzz.token_set_ratio, workers=1)[0], np.float32) / 100.0)
    empty = np.array([], dtype=np.float32)
    return (rows_chunk,
            np.concatenate(wr) if wr else empty, np.concatenate(ts) if ts else empty,
            np.concatenate(tt) if tt else empty, np.concatenate(pa) if pa else empty,
            np.concatenate(lv) if lv else empty, np.concatenate(ats) if ats else empty,
            np.concatenate(ate) if ate else empty)


In [ ]:
# 5. Process Shard
def process_single_shard(s_idx):
    shard_size = max(1, -(-total_s1 // NUM_SHARDS))
    start_idx = s_idx * shard_size
    end_idx = min(total_s1, (s_idx + 1) * shard_size)

    s1_shard = s1_full.iloc[start_idx:end_idx].reset_index(drop=True)
    print(f"
{'='*60}
PROCESSING SHARD {s_idx} ({len(s1_shard):,} queries: rows {start_idx:,} to {end_idx:,})
{'='*60}")

    t0 = time.time()
    names = [norm_s1_row(t) for t in s1_shard["business_name"]]
    s1_shard["name_full"]  = [x[0] for x in names]
    s1_shard["name_core"]  = [x[1] for x in names]
    s1_shard["name_trade"] = [x[2] for x in names]
    s1_shard["addr_norm"]  = s1_shard["business_address"].str.lower().str.replace(r"[^a-z0-9\s]", " ", regex=True).str.strip()
    s1_shard["combined"]   = s1_shard["name_core"] + " " + s1_shard["addr_norm"]
    s1_shard["country_norm"] = s1_shard["country"].map(lambda x: x.strip().lower() if x.strip() else "__unknown__")

    # 1. Blocking
    t_b = time.time()
    n_i, n_c, n_v = run_fast_country_blocking(s1_shard, "name", target_meta["vec_name"], BLOCK_TOP_K, BLOCK_MIN_SIM)
    a_i, a_c, a_v = run_fast_country_blocking(s1_shard, "addr", target_meta["vec_addr"], BLOCK_TOP_K // 2, BLOCK_MIN_SIM)
    c_i, c_c, c_v = run_fast_country_blocking(s1_shard, "comb", target_meta["vec_comb"], BLOCK_TOP_K // 2, BLOCK_MIN_SIM)

    n23 = len(target_meta["entity_ids"])
    def make_key(i1, i23): return i1.astype(np.int64) * (n23 + 1) + i23.astype(np.int64)

    df_n = pd.DataFrame({"key": make_key(n_i, n_c), "nsim": n_v})
    df_a = pd.DataFrame({"key": make_key(a_i, a_c), "asim": a_v})
    df_c = pd.DataFrame({"key": make_key(c_i, c_c), "csim": c_v})

    cand = df_n.merge(df_a, on="key", how="outer").merge(df_c, on="key", how="outer")
    for col in ["nsim", "asim", "csim"]: cand[col] = cand[col].fillna(0.0).astype(np.float32)

    cand["s1i"]  = (cand["key"] // (n23 + 1)).astype(np.int32)
    cand["s23i"] = (cand["key"] %  (n23 + 1)).astype(np.int32)
    cand = cand.drop(columns="key")

    cand["s1_id"]  = s1_shard["entity_id"].values[cand["s1i"].values]
    cand["s23_id"] = target_meta["entity_ids"][cand["s23i"].values]
    print(f"  [Shard {s_idx}] Blocking complete: {len(cand):,} pairs in {time.time()-t_b:.1f}s")

    # 2. Features
    t_f = time.time()
    s1i, s23i = cand["s1i"].values, cand["s23i"].values
    nc1, nc2 = s1_shard["name_core"].values[s1i], target_meta["name_core"][s23i]
    nf1, nf2 = s1_shard["name_full"].values[s1i], target_meta["name_full"][s23i]
    a1,  a2  = s1_shard["addr_norm"].values[s1i],  target_meta["addr_norm"][s23i]
    c1,  c2  = s1_shard["country"].values[s1i],    target_meta["country_raw"][s23i]
    s23_ids = cand["s23_id"].values
    n = len(cand)

    feat = {}
    feat["nsim"] = cand["nsim"].values.astype(np.float32)
    feat["asim"] = cand["asim"].values.astype(np.float32)
    feat["csim"] = cand["csim"].values.astype(np.float32)
    feat["max_sim"] = np.maximum(np.maximum(feat["nsim"], feat["asim"]), feat["csim"])

    n_inter, n_sz1, n_sz2 = _binary_intersection(s1_shard["name_core"].values, target_meta["name_core"], s1i, s23i, analyzer="word", token_pattern=r"\S+")
    feat["name_jaccard_tok"], feat["name_overlap"] = _jac_ov(n_inter, n_sz1, n_sz2)
    n3i, n3s1, n3s2 = _binary_intersection(s1_shard["name_core"].values, target_meta["name_core"], s1i, s23i, analyzer="char", ngram_range=(3, 3))
    feat["name_jac3"], _ = _jac_ov(n3i, n3s1, n3s2)
    n4i, n4s1, n4s2 = _binary_intersection(s1_shard["name_core"].values, target_meta["name_core"], s1i, s23i, analyzer="char", ngram_range=(4, 4))
    feat["name_jac4"], _ = _jac_ov(n4i, n4s1, n4s2)

    a_inter, a_sz1, a_sz2 = _binary_intersection(s1_shard["addr_norm"].values, target_meta["addr_norm"], s1i, s23i, analyzer="word", token_pattern=r"\S+")
    feat["addr_jaccard_tok"], feat["addr_overlap"] = _jac_ov(a_inter, a_sz1, a_sz2)
    a3i, a3s1, a3s2 = _binary_intersection(s1_shard["addr_norm"].values, target_meta["addr_norm"], s1i, s23i, analyzer="char", ngram_range=(3, 3))
    feat["addr_jac3"], _ = _jac_ov(a3i, a3s1, a3s2)
    num_inter, _, _ = _binary_intersection(s1_shard["addr_norm"].values, target_meta["addr_norm"], s1i, s23i, analyzer="word", token_pattern=r"\d+")
    feat["addr_num_overlap"] = num_inter

    d1i, d1s1, d1s2 = _binary_intersection(s1_shard["name_trade"].values, target_meta["name_core"], s1i, s23i, analyzer="word", token_pattern=r"\S+")
    d1_j, _ = _jac_ov(d1i, d1s1, d1s2)
    d2i, d2s1, d2s2 = _binary_intersection(s1_shard["name_core"].values, target_meta["name_trade"], s1i, s23i, analyzer="word", token_pattern=r"\S+")
    d2_j, _ = _jac_ov(d2i, d2s1, d2s2)
    feat["dba_trade_jaccard"] = np.maximum(feat["name_jaccard_tok"], np.maximum(d1_j, d2_j)).astype(np.float32)

    feat["name_exact"] = ((nf1 == nf2) & (nf1 != "")).astype(np.float32)
    nlen1 = np.array([len(x) for x in nc1], dtype=np.float32)
    nlen2 = np.array([len(x) for x in nc2], dtype=np.float32)
    feat["name_len_diff"] = np.abs(nlen1 - nlen2)
    maxn = np.maximum(nlen1, nlen2)
    feat["name_len_ratio"] = np.divide(np.minimum(nlen1, nlen2), maxn, out=np.zeros_like(nlen1), where=maxn > 0)
    feat["name_tok_diff"] = np.abs(n_sz1 - n_sz2)
    ft1 = np.array([x.split()[0] if x.split() else "" for x in nc1])
    ft2 = np.array([x.split()[0] if x.split() else "" for x in nc2])
    feat["name_first_tok"] = ((ft1 == ft2) & (ft1 != "")).astype(np.float32)

    alen1 = np.array([len(x) for x in a1], dtype=np.float32)
    alen2 = np.array([len(x) for x in a2], dtype=np.float32)
    feat["addr_missing"] = ((alen1 == 0) | (alen2 == 0)).astype(np.float32)
    maxa = np.maximum(alen1, alen2)
    feat["addr_len_ratio"] = np.divide(np.minimum(alen1, alen2), maxa, out=np.zeros_like(alen1), where=maxa > 0)

    feat["country_match"] = ((c1 == c2) & (c1 != "")).astype(np.float32)
    feat["country_mismatch"] = ((c1 != c2) & (c1 != "") & (c2 != "")).astype(np.float32)
    feat["src_is_s2"] = np.array([sid.startswith("S2-") for sid in s23_ids], dtype=np.float32)
    feat["name_x_addr"] = feat["name_jaccard_tok"] * feat["addr_jaccard_tok"]

    # Parallel RapidFuzz
    wratio = np.zeros(n, np.float32); tsort = np.zeros(n, np.float32)
    tset = np.zeros(n, np.float32); partial = np.zeros(n, np.float32); levn = np.zeros(n, np.float32)
    addr_ts = np.zeros(n, np.float32); addr_te = np.zeros(n, np.float32)

    order = np.argsort(s1i, kind="stable")
    sorted_s1i = s1i[order]
    _, g_start = np.unique(sorted_s1i, return_index=True)
    bounds = np.append(g_start, n)
    n_groups = len(g_start)

    edges = np.linspace(0, n_groups, N_CORES + 1).round().astype(np.int64)
    tasks = []
    for gi in range(N_CORES):
        g_lo, g_hi = int(edges[gi]), int(edges[gi + 1])
        if g_lo >= g_hi: continue
        p_lo, p_hi = int(bounds[g_lo]), int(bounds[g_hi])
        r_chunk = order[p_lo:p_hi]
        lb = bounds[g_lo:g_hi + 1] - p_lo
        tasks.append((r_chunk, nc1[r_chunk], nc2[r_chunk], a1[r_chunk], a2[r_chunk], list(zip(lb[:-1], lb[1:]))))

    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
    with ProcessPoolExecutor(max_workers=N_CORES, mp_context=ctx) as ex:
        futs = {ex.submit(_rf_worker, t): i for i, t in enumerate(tasks)}
        for fut in as_completed(futs):
            r_chunk, wr, ts_, tt, pa, lv, ats, ate = fut.result()
            wratio[r_chunk] = wr; tsort[r_chunk] = ts_; tset[r_chunk] = tt
            partial[r_chunk] = pa; levn[r_chunk] = lv
            addr_ts[r_chunk] = ats; addr_te[r_chunk] = ate

    feat.update(dict(name_wratio=wratio, name_tsort=tsort, name_tset=tset, name_partial=partial, name_lev_norm=levn, addr_tsort=addr_ts, addr_tset=addr_te))

    tmp = pd.DataFrame({"s1_id": cand["s1_id"].values, "s23_id": cand["s23_id"].values, "max_sim": feat["max_sim"]})
    tmp["rank_in_cand"] = tmp.groupby("s1_id")["max_sim"].rank(ascending=False, method="first")
    tmp["top_sim"] = tmp.groupby("s1_id")["max_sim"].transform("max")
    tmp["n_cand_for_entity"] = tmp.groupby("s1_id")["max_sim"].transform("size")
    tmp["reverse_rank"] = tmp.groupby("s23_id")["max_sim"].rank(ascending=False, method="first")

    feat["rank_in_cand"] = tmp["rank_in_cand"].values.astype(np.float32)
    feat["gap_to_top"] = (tmp["top_sim"] - tmp["max_sim"]).values.astype(np.float32)
    feat["n_cand_for_entity"] = tmp["n_cand_for_entity"].values.astype(np.float32)
    feat["reverse_rank"] = tmp["reverse_rank"].values.astype(np.float32)
    feat["mutual_top1"] = ((tmp["rank_in_cand"] == 1) & (tmp["reverse_rank"] == 1)).values.astype(np.float32)

    s1_nf = s1_shard["name_core"].value_counts()
    s1_af = s1_shard["addr_norm"].value_counts()
    feat["name_freq_s1"] = pd.Series(nc1).map(s1_nf).fillna(0).values.astype(np.float32)
    feat["addr_freq_s1"] = pd.Series(a1).map(s1_af).fillna(0).values.astype(np.float32)

    feat_df = pd.DataFrame(feat)[feature_names]
    print(f"  [Shard {s_idx}] Features engineered in {time.time()-t_f:.1f}s. Shape: {feat_df.shape}")

    # 3. LightGBM Scoring + Calibration
    X = feat_df.values.astype(np.float32)
    raw_scores = np.mean([m.predict(X) for m in models], axis=0)
    cal_scores = iso.transform(raw_scores)
    cand["score"] = cal_scores

    # 4. Two-Threshold Decision Rule
    match_dict, cand_dict = {}, {}
    grouped = cand.groupby("s1_id")
    for sid, grp in grouped:
        grp_s = grp.sort_values("score", ascending=False)
        ids = grp_s["s23_id"].tolist()
        scs = grp_s["score"].tolist()
        cand_dict[sid] = list(dict.fromkeys(ids))
        if scs and scs[0] >= t_first:
            keep = [i for i, s in zip(ids, scs) if s >= t_rest]
            match_dict[sid] = list(dict.fromkeys(keep))
        else:
            match_dict[sid] = []

    for sid in s1_shard["entity_id"]:
        if sid not in match_dict:
            match_dict[sid] = []
            cand_dict[sid] = []

    out_file = os.path.join(OUTPUT_SHARD_DIR, f"shard_{s_idx}_results.pkl")
    with open(out_file, "wb") as f:
        pickle.dump({"matches": match_dict, "cands": cand_dict}, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  [Shard {s_idx}] SUCCESS in {time.time()-t0:.1f}s -> Saved {out_file}")

# Execute
if RUN_ALL_SHARDS:
    for si in range(NUM_SHARDS):
        process_single_shard(si)
else:
    process_single_shard(SHARD_IDX)
